In [1]:
import os
from pathlib import Path
import sqlite3

# 1. Định nghĩa thư mục nguồn và danh sách các file cần kiểm tra
source_dir = Path('/content')
required_files = [
    'coffee_bi.db',
    'sales_clean.csv',
    'inventory_weekly.csv',
    'waste_log.csv',
    'bom.csv'
]

# 2. Kiểm tra sự tồn tại và kích thước của từng file
missing_files = []
print("=== KIỂM TRA TẬP TIN NGUỒN ===")
for file_name in required_files:
    file_path = source_dir / file_name
    if file_path.exists():
        size_kb = file_path.stat().st_size / 1024
        print(f"[OK] {file_name} - Kích thước: {size_kb:.2f} KB")
    else:
        missing_files.append(file_name)

# Dừng và thông báo nếu thiếu file
if missing_files:
    print("\n[ERROR] Thiếu các file sau trong thư mục:")
    for f in missing_files:
        print(f" - {f}")
    raise FileNotFoundError("Vui lòng tải lên đầy đủ các file trước khi tiếp tục!")
else:
    print("\n[SUCCESS] Đã tìm thấy đầy đủ 5 file cần thiết.\n")

# 3. Kết nối sqlite3 ở chế độ chỉ đọc và truy vấn cấu trúc DB
db_path = source_dir / 'coffee_bi.db'
# Sử dụng URI với mode=ro để mở database ở chế độ chỉ đọc
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
cursor = conn.cursor()

print("=== THÔNG TIN CÁC BẢNG TRONG DATABASE ===")
try:
    # Lấy danh sách các bảng
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]

    for table in tables:
        # Lấy số lượng dòng
        cursor.execute(f"SELECT COUNT(*) FROM `{table}`")
        row_count = cursor.fetchone()[0]

        # Lấy tên các cột
        cursor.execute(f"PRAGMA table_info(`{table}`)")
        columns = [col[1] for col in cursor.fetchall()]

        print(f"Bảng: {table}")
        print(f"  - Số dòng: {row_count}")
        print(f"  - Các cột: {', '.join(columns)}")
        print("-" * 40)
except sqlite3.Error as e:
    print(f"Có lỗi xảy ra khi truy vấn database: {e}")


=== KIỂM TRA TẬP TIN NGUỒN ===
[OK] coffee_bi.db - Kích thước: 10524.00 KB
[OK] sales_clean.csv - Kích thước: 25161.00 KB
[OK] inventory_weekly.csv - Kích thước: 229.00 KB
[OK] waste_log.csv - Kích thước: 632.77 KB
[OK] bom.csv - Kích thước: 14.99 KB

[SUCCESS] Đã tìm thấy đầy đủ 5 file cần thiết.

=== THÔNG TIN CÁC BẢNG TRONG DATABASE ===
Bảng: Dim_Date
  - Số dòng: 187
  - Các cột: date_key, full_date, year, quarter, month, month_name, day_of_week, day_name, is_weekend, week_start, week_no, in_data_period
----------------------------------------
Bảng: Dim_Store
  - Số dòng: 3
  - Các cột: store_key, store_id, store_location
----------------------------------------
Bảng: Dim_Shift
  - Số dòng: 3
  - Các cột: shift_key, shift_name, start_hour, end_hour
----------------------------------------
Bảng: Dim_Menu_Item
  - Số dòng: 41
  - Các cột: menu_item_key, product_id, product_category, product_type, product_detail, base_item, size
----------------------------------------
Bảng: Dim_Ingre

In [2]:
import csv
from decimal import Decimal

# 1. Định nghĩa cấu trúc các bảng TEMP và thực hiện DROP nếu đã tồn tại để đảm bảo tính idempotent
temp_tables = {
    "src_sales": """
        CREATE TEMP TABLE src_sales (
            transaction_id TEXT,
            transaction_date TEXT,
            store_id INTEGER,
            product_id TEXT,
            shift TEXT,
            transaction_qty INTEGER,
            is_drink INTEGER,
            revenue_cents INTEGER
        );
    """,
    "src_inventory": """
        CREATE TEMP TABLE src_inventory (
            store_id INTEGER,
            ingredient_id TEXT,
            week_start TEXT,
            theoretical_usage REAL
        );
    """,
    "src_waste": """
        CREATE TEMP TABLE src_waste (
            waste_id TEXT
        );
    """,
    "src_bom": """
        CREATE TEMP TABLE src_bom (
            product_id TEXT,
            ingredient_id TEXT
        );
    """
}

# Khởi tạo/tái tạo các bảng TEMP
for table_name, create_sql in temp_tables.items():
    cursor.execute(f"DROP TABLE IF EXISTS temp.{table_name};")
    cursor.execute(create_sql)
conn.commit()

# 2. Đọc và nạp dữ liệu từ các file CSV bằng utf-8-sig và truy vấn có tham số

# --- Nạp src_sales ---
sales_path = source_dir / 'sales_clean.csv'
with open(sales_path, mode='r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    sales_data = []
    for row in reader:
        # Tính toán revenue_cents bằng Decimal
        revenue_val = row.get('revenue', '0')
        revenue_cents = int(Decimal(revenue_val) * 100) if revenue_val else 0

        sales_data.append((
            row.get('transaction_id'),
            row.get('transaction_date'),
            int(row.get('store_id')) if row.get('store_id') else None,
            row.get('product_id'),
            row.get('shift'),
            int(row.get('transaction_qty')) if row.get('transaction_qty') else 0,
            int(row.get('is_drink')) if row.get('is_drink') else 0,
            revenue_cents
        ))

    cursor.executemany("""
        INSERT INTO temp.src_sales
        (transaction_id, transaction_date, store_id, product_id, shift, transaction_qty, is_drink, revenue_cents)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?);
    """, sales_data)

# --- Nạp src_inventory ---
inventory_path = source_dir / 'inventory_weekly.csv'
with open(inventory_path, mode='r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    inventory_data = []
    for row in reader:
        inventory_data.append((
            int(row.get('store_id')) if row.get('store_id') else None,
            row.get('ingredient_id'),
            row.get('week_start'),
            float(row.get('theoretical_usage')) if row.get('theoretical_usage') else 0.0
        ))

    cursor.executemany("""
        INSERT INTO temp.src_inventory
        (store_id, ingredient_id, week_start, theoretical_usage)
        VALUES (?, ?, ?, ?);
    """, inventory_data)

# --- Nạp src_waste ---
waste_path = source_dir / 'waste_log.csv'
with open(waste_path, mode='r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    waste_data = []
    for row in reader:
        waste_data.append((row.get('waste_id'),))

    cursor.executemany("""
        INSERT INTO temp.src_waste (waste_id) VALUES (?);
    """, waste_data)

# --- Nạp src_bom ---
bom_path = source_dir / 'bom.csv'
with open(bom_path, mode='r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    bom_data = []
    for row in reader:
        bom_data.append((
            row.get('product_id'),
            row.get('ingredient_id')
        ))

    cursor.executemany("""
        INSERT INTO temp.src_bom (product_id, ingredient_id) VALUES (?, ?);
    """, bom_data)

conn.commit()

# 3. Thống kê và in kết quả kiểm thử số lượng dòng nạp được
print("=== THỐNG KÊ SỐ DÒNG CỦA 4 BẢNG TẠM ===")
for table_name in temp_tables.keys():
    cursor.execute(f"SELECT COUNT(*) FROM temp.{table_name};")
    count = cursor.fetchone()[0]
    print(f"Bảng tạm `{table_name}`: {count} dòng.")

print("\n=== THỐNG KÊ CHI TIẾT BẢNG TẠM src_sales ===")
# Thống kê cho toàn bộ POS
cursor.execute("""
    SELECT COUNT(*), SUM(transaction_qty), SUM(revenue_cents)
    FROM temp.src_sales;
""")
total_rows, total_qty, total_rev_cents = cursor.fetchone()

# Thống kê cho tập is_drink = 1
cursor.execute("""
    SELECT COUNT(*), SUM(transaction_qty), SUM(revenue_cents)
    FROM temp.src_sales
    WHERE is_drink = 1;
""")
drink_rows, drink_qty, drink_rev_cents = cursor.fetchone()

print("1. Toàn bộ danh mục (Toàn bộ dữ liệu POS):")
print(f"  - Số dòng: {total_rows}")
print(f"  - Tổng sản lượng (transaction_qty): {total_qty}")
print(f"  - Tổng doanh thu: {total_rev_cents / 100:.2f} USD ({total_rev_cents} cents)")

print("\n2. Phân khúc Đồ uống (is_drink = 1):")
print(f"  - Số dòng: {drink_rows}")
print(f"  - Tổng sản lượng (transaction_qty): {drink_qty}")
print(f"  - Tổng doanh thu: {drink_rev_cents / 100:.2f} USD ({drink_rev_cents} cents)")


=== THỐNG KÊ SỐ DÒNG CỦA 4 BẢNG TẠM ===
Bảng tạm `src_sales`: 149116 dòng.
Bảng tạm `src_inventory`: 1755 dòng.
Bảng tạm `src_waste`: 5042 dòng.
Bảng tạm `src_bom`: 178 dòng.

=== THỐNG KÊ CHI TIẾT BẢNG TẠM src_sales ===
1. Toàn bộ danh mục (Toàn bộ dữ liệu POS):
  - Số dòng: 149116
  - Tổng sản lượng (transaction_qty): 214470
  - Tổng doanh thu: 698812.33 USD (69881233 cents)

2. Phân khúc Đồ uống (is_drink = 1):
  - Số dòng: 115333
  - Tổng sản lượng (transaction_qty): 176444
  - Tổng doanh thu: 538774.40 USD (53877440 cents)


In [3]:
import csv

output_dir = Path("/content/ban_giao_44")
sql_dir = output_dir / "sql"
sql_dir.mkdir(parents=True, exist_ok=True)
full_sql_text = "-- SQLite. Use run_audit.py to load independent CSV references into TEMP tables.\n-- All seven statements read the existing target without changing it.\n\n-- T01\nWITH checks AS (\nSELECT 'POS toàn bộ' AS scope, (SELECT COUNT(*) FROM src_sales) AS expected, (SELECT COUNT(*) FROM Fact_Sales) AS actual\nUNION ALL\nSELECT 'NB04: 41 đồ uống' AS scope, (SELECT COUNT(*) FROM src_sales WHERE is_drink = 1) AS expected, (SELECT COUNT(*) FROM Fact_Sales) AS actual\nUNION ALL\nSELECT 'Fact_Inventory_Audit', (SELECT COUNT(*) FROM src_inventory), (SELECT COUNT(*) FROM Fact_Inventory_Audit)\nUNION ALL\nSELECT 'Fact_Waste_Log', (SELECT COUNT(*) FROM src_waste), (SELECT COUNT(*) FROM Fact_Waste_Log)\nUNION ALL\nSELECT 'Bridge_BOM', (SELECT COUNT(*) FROM src_bom), (SELECT COUNT(*) FROM Bridge_BOM)\n)\nSELECT 'T01' AS test, scope, expected, actual, CASE WHEN expected = actual THEN 'PASS' ELSE 'FAIL' END AS status FROM checks;\n\n-- T02 (money in integer cents)\nWITH checks AS (\nSELECT 'POS toàn bộ' AS scope, (SELECT SUM(revenue_cents) FROM src_sales) AS expected, (SELECT SUM(CAST(ROUND(revenue * 100) AS INTEGER)) FROM Fact_Sales) AS actual\nUNION ALL\nSELECT 'NB04: 41 đồ uống' AS scope, (SELECT SUM(revenue_cents) FROM src_sales WHERE is_drink = 1) AS expected, (SELECT SUM(CAST(ROUND(revenue * 100) AS INTEGER)) FROM Fact_Sales) AS actual\n)\nSELECT 'T02' AS test, scope, expected, actual, CASE WHEN expected = actual THEN 'PASS' ELSE 'FAIL' END AS status FROM checks;\n\n-- T03\nWITH checks AS (\nSELECT 'POS toàn bộ' AS scope, (SELECT SUM(transaction_qty) FROM src_sales) AS expected, (SELECT SUM(transaction_qty) FROM Fact_Sales) AS actual\nUNION ALL\nSELECT 'NB04: 41 đồ uống' AS scope, (SELECT SUM(transaction_qty) FROM src_sales WHERE is_drink = 1) AS expected, (SELECT SUM(transaction_qty) FROM Fact_Sales) AS actual\n)\nSELECT 'T03' AS test, scope, expected, actual, CASE WHEN expected = actual THEN 'PASS' ELSE 'FAIL' END AS status FROM checks;\n\n-- T04\nWITH checks AS (\nSELECT 'Fact_Sales.date_key' AS scope, COUNT(*) AS actual FROM Fact_Sales f WHERE f.date_key IS NULL\nUNION ALL\nSELECT 'Fact_Sales.store_key' AS scope, COUNT(*) AS actual FROM Fact_Sales f WHERE f.store_key IS NULL\nUNION ALL\nSELECT 'Fact_Sales.shift_key' AS scope, COUNT(*) AS actual FROM Fact_Sales f WHERE f.shift_key IS NULL\nUNION ALL\nSELECT 'Fact_Sales.menu_item_key' AS scope, COUNT(*) AS actual FROM Fact_Sales f WHERE f.menu_item_key IS NULL\nUNION ALL\nSELECT 'Fact_Inventory_Audit.week_start_date_key' AS scope, COUNT(*) AS actual FROM Fact_Inventory_Audit f WHERE f.week_start_date_key IS NULL\nUNION ALL\nSELECT 'Fact_Inventory_Audit.store_key' AS scope, COUNT(*) AS actual FROM Fact_Inventory_Audit f WHERE f.store_key IS NULL\nUNION ALL\nSELECT 'Fact_Inventory_Audit.ingredient_key' AS scope, COUNT(*) AS actual FROM Fact_Inventory_Audit f WHERE f.ingredient_key IS NULL\nUNION ALL\nSELECT 'Fact_Waste_Log.date_key' AS scope, COUNT(*) AS actual FROM Fact_Waste_Log f WHERE f.date_key IS NULL\nUNION ALL\nSELECT 'Fact_Waste_Log.store_key' AS scope, COUNT(*) AS actual FROM Fact_Waste_Log f WHERE f.store_key IS NULL\nUNION ALL\nSELECT 'Fact_Waste_Log.shift_key' AS scope, COUNT(*) AS actual FROM Fact_Waste_Log f WHERE f.shift_key IS NULL\nUNION ALL\nSELECT 'Fact_Waste_Log.ingredient_key' AS scope, COUNT(*) AS actual FROM Fact_Waste_Log f WHERE f.ingredient_key IS NULL\nUNION ALL\nSELECT 'Bridge_BOM.menu_item_key' AS scope, COUNT(*) AS actual FROM Bridge_BOM f WHERE f.menu_item_key IS NULL\nUNION ALL\nSELECT 'Bridge_BOM.ingredient_key' AS scope, COUNT(*) AS actual FROM Bridge_BOM f WHERE f.ingredient_key IS NULL\n)\nSELECT 'T04' AS test, scope, 0 AS expected, actual, CASE WHEN actual = 0 THEN 'PASS' ELSE 'FAIL' END AS status FROM checks;\n\n-- T05\nWITH checks AS (\nSELECT 'Fact_Sales.date_key' AS scope, COUNT(*) AS actual FROM Fact_Sales f LEFT JOIN Dim_Date d ON f.date_key = d.date_key WHERE f.date_key IS NOT NULL AND d.date_key IS NULL\nUNION ALL\nSELECT 'Fact_Sales.store_key' AS scope, COUNT(*) AS actual FROM Fact_Sales f LEFT JOIN Dim_Store d ON f.store_key = d.store_key WHERE f.store_key IS NOT NULL AND d.store_key IS NULL\nUNION ALL\nSELECT 'Fact_Sales.shift_key' AS scope, COUNT(*) AS actual FROM Fact_Sales f LEFT JOIN Dim_Shift d ON f.shift_key = d.shift_key WHERE f.shift_key IS NOT NULL AND d.shift_key IS NULL\nUNION ALL\nSELECT 'Fact_Sales.menu_item_key' AS scope, COUNT(*) AS actual FROM Fact_Sales f LEFT JOIN Dim_Menu_Item d ON f.menu_item_key = d.menu_item_key WHERE f.menu_item_key IS NOT NULL AND d.menu_item_key IS NULL\nUNION ALL\nSELECT 'Fact_Inventory_Audit.week_start_date_key' AS scope, COUNT(*) AS actual FROM Fact_Inventory_Audit f LEFT JOIN Dim_Date d ON f.week_start_date_key = d.date_key WHERE f.week_start_date_key IS NOT NULL AND d.date_key IS NULL\nUNION ALL\nSELECT 'Fact_Inventory_Audit.store_key' AS scope, COUNT(*) AS actual FROM Fact_Inventory_Audit f LEFT JOIN Dim_Store d ON f.store_key = d.store_key WHERE f.store_key IS NOT NULL AND d.store_key IS NULL\nUNION ALL\nSELECT 'Fact_Inventory_Audit.ingredient_key' AS scope, COUNT(*) AS actual FROM Fact_Inventory_Audit f LEFT JOIN Dim_Ingredient d ON f.ingredient_key = d.ingredient_key WHERE f.ingredient_key IS NOT NULL AND d.ingredient_key IS NULL\nUNION ALL\nSELECT 'Fact_Waste_Log.date_key' AS scope, COUNT(*) AS actual FROM Fact_Waste_Log f LEFT JOIN Dim_Date d ON f.date_key = d.date_key WHERE f.date_key IS NOT NULL AND d.date_key IS NULL\nUNION ALL\nSELECT 'Fact_Waste_Log.store_key' AS scope, COUNT(*) AS actual FROM Fact_Waste_Log f LEFT JOIN Dim_Store d ON f.store_key = d.store_key WHERE f.store_key IS NOT NULL AND d.store_key IS NULL\nUNION ALL\nSELECT 'Fact_Waste_Log.shift_key' AS scope, COUNT(*) AS actual FROM Fact_Waste_Log f LEFT JOIN Dim_Shift d ON f.shift_key = d.shift_key WHERE f.shift_key IS NOT NULL AND d.shift_key IS NULL\nUNION ALL\nSELECT 'Fact_Waste_Log.ingredient_key' AS scope, COUNT(*) AS actual FROM Fact_Waste_Log f LEFT JOIN Dim_Ingredient d ON f.ingredient_key = d.ingredient_key WHERE f.ingredient_key IS NOT NULL AND d.ingredient_key IS NULL\nUNION ALL\nSELECT 'Bridge_BOM.menu_item_key' AS scope, COUNT(*) AS actual FROM Bridge_BOM f LEFT JOIN Dim_Menu_Item d ON f.menu_item_key = d.menu_item_key WHERE f.menu_item_key IS NOT NULL AND d.menu_item_key IS NULL\nUNION ALL\nSELECT 'Bridge_BOM.ingredient_key' AS scope, COUNT(*) AS actual FROM Bridge_BOM f LEFT JOIN Dim_Ingredient d ON f.ingredient_key = d.ingredient_key WHERE f.ingredient_key IS NOT NULL AND d.ingredient_key IS NULL\n)\nSELECT 'T05' AS test, scope, 0 AS expected, actual, CASE WHEN actual = 0 THEN 'PASS' ELSE 'FAIL' END AS status FROM checks;\n\n-- T06\nWITH checks AS (\nSELECT 'Dim_Date (date_key)' AS scope, COUNT(*) AS actual FROM (SELECT date_key FROM Dim_Date GROUP BY date_key HAVING COUNT(*) > 1 OR date_key IS NULL)\nUNION ALL\nSELECT 'Dim_Store (store_key)' AS scope, COUNT(*) AS actual FROM (SELECT store_key FROM Dim_Store GROUP BY store_key HAVING COUNT(*) > 1 OR store_key IS NULL)\nUNION ALL\nSELECT 'Dim_Shift (shift_key)' AS scope, COUNT(*) AS actual FROM (SELECT shift_key FROM Dim_Shift GROUP BY shift_key HAVING COUNT(*) > 1 OR shift_key IS NULL)\nUNION ALL\nSELECT 'Dim_Menu_Item (menu_item_key)' AS scope, COUNT(*) AS actual FROM (SELECT menu_item_key FROM Dim_Menu_Item GROUP BY menu_item_key HAVING COUNT(*) > 1 OR menu_item_key IS NULL)\nUNION ALL\nSELECT 'Dim_Ingredient (ingredient_key)' AS scope, COUNT(*) AS actual FROM (SELECT ingredient_key FROM Dim_Ingredient GROUP BY ingredient_key HAVING COUNT(*) > 1 OR ingredient_key IS NULL)\nUNION ALL\nSELECT 'Bridge_BOM (bom_key)' AS scope, COUNT(*) AS actual FROM (SELECT bom_key FROM Bridge_BOM GROUP BY bom_key HAVING COUNT(*) > 1 OR bom_key IS NULL)\nUNION ALL\nSELECT 'Fact_Sales (sales_key)' AS scope, COUNT(*) AS actual FROM (SELECT sales_key FROM Fact_Sales GROUP BY sales_key HAVING COUNT(*) > 1 OR sales_key IS NULL)\nUNION ALL\nSELECT 'Fact_Inventory_Audit (inventory_key)' AS scope, COUNT(*) AS actual FROM (SELECT inventory_key FROM Fact_Inventory_Audit GROUP BY inventory_key HAVING COUNT(*) > 1 OR inventory_key IS NULL)\nUNION ALL\nSELECT 'Fact_Waste_Log (waste_key)' AS scope, COUNT(*) AS actual FROM (SELECT waste_key FROM Fact_Waste_Log GROUP BY waste_key HAVING COUNT(*) > 1 OR waste_key IS NULL)\nUNION ALL\nSELECT 'Fact_Sales (transaction_id)' AS scope, COUNT(*) AS actual FROM (SELECT transaction_id FROM Fact_Sales GROUP BY transaction_id HAVING COUNT(*) > 1 OR transaction_id IS NULL)\nUNION ALL\nSELECT 'Fact_Waste_Log (waste_id)' AS scope, COUNT(*) AS actual FROM (SELECT waste_id FROM Fact_Waste_Log GROUP BY waste_id HAVING COUNT(*) > 1 OR waste_id IS NULL)\nUNION ALL\nSELECT 'Fact_Inventory_Audit (store_key, ingredient_key, week_start_date_key)' AS scope, COUNT(*) AS actual FROM (SELECT store_key, ingredient_key, week_start_date_key FROM Fact_Inventory_Audit GROUP BY store_key, ingredient_key, week_start_date_key HAVING COUNT(*) > 1 OR store_key IS NULL OR ingredient_key IS NULL OR week_start_date_key IS NULL)\nUNION ALL\nSELECT 'Bridge_BOM (menu_item_key, ingredient_key, bom_version)' AS scope, COUNT(*) AS actual FROM (SELECT menu_item_key, ingredient_key, bom_version FROM Bridge_BOM GROUP BY menu_item_key, ingredient_key, bom_version HAVING COUNT(*) > 1 OR menu_item_key IS NULL OR ingredient_key IS NULL OR bom_version IS NULL)\n)\nSELECT 'T06' AS test, scope, 0 AS expected, actual, CASE WHEN actual = 0 THEN 'PASS' ELSE 'FAIL' END AS status FROM checks;\n\n-- T07. Compare every source/target grain and totals separately by ingredient.\nWITH target_rows AS (\n SELECT s.store_id, i.ingredient_id, d.full_date AS week_start,\n        f.theoretical_usage AS tu\n FROM Fact_Inventory_Audit f\n LEFT JOIN Dim_Store s ON s.store_key = f.store_key\n LEFT JOIN Dim_Ingredient i ON i.ingredient_key = f.ingredient_key\n LEFT JOIN Dim_Date d ON d.date_key = f.week_start_date_key\n), source_groups AS (\n SELECT store_id, ingredient_id, week_start, COUNT(*) AS n, SUM(theoretical_usage) AS tu\n FROM src_inventory GROUP BY store_id, ingredient_id, week_start\n), target_groups AS (\n SELECT store_id, ingredient_id, week_start, COUNT(*) AS n, SUM(tu) AS tu\n FROM target_rows GROUP BY store_id, ingredient_id, week_start\n), keys AS (\n SELECT store_id, ingredient_id, week_start FROM source_groups\n UNION SELECT store_id, ingredient_id, week_start FROM target_groups\n), compared AS (\n SELECT k.ingredient_id, s.tu AS source_tu, t.tu AS target_tu,\n        CASE WHEN k.store_id IS NULL OR k.ingredient_id IS NULL OR k.week_start IS NULL\n          OR s.n IS NULL OR t.n IS NULL OR s.n <> 1 OR t.n <> 1\n          OR s.tu IS NULL OR t.tu IS NULL OR ABS(s.tu - t.tu) > 0.000001\n          THEN 1 ELSE 0 END AS bad\n FROM keys k\n LEFT JOIN source_groups s USING (store_id, ingredient_id, week_start)\n LEFT JOIN target_groups t USING (store_id, ingredient_id, week_start)\n), totals AS (\n SELECT ingredient_id, SUM(source_tu) AS expected, SUM(target_tu) AS actual, SUM(bad) AS bad_grains\n FROM compared GROUP BY ingredient_id\n)\nSELECT 'T07' AS test, COALESCE(ingredient_id, 'UNMAPPED') AS scope,\n       ROUND(expected, 6) AS expected, ROUND(actual, 6) AS actual,\n       CASE WHEN bad_grains = 0 AND ABS(expected - actual) <= 0.000001 THEN 'PASS' ELSE 'FAIL' END AS status,\n       bad_grains\nFROM totals ORDER BY scope;\n\n"

statements = []
buffer = ""
for line in full_sql_text.splitlines(keepends=True):
    buffer += line
    if sqlite3.complete_statement(buffer):
        statements.append(buffer)
        buffer = ""
if len(statements) != 7 or buffer.strip():
    raise ValueError("Cần đúng 7 câu SQL hoàn chỉnh")

results = []
for statement in statements:
    result = conn.execute(statement)
    columns = [column[0] for column in result.description]
    results.extend(dict(zip(columns, row)) for row in result.fetchall())

expected_counts = {"T01": 5, "T02": 2, "T03": 2, "T04": 13,
                   "T05": 13, "T06": 13, "T07": 22}
counts = {test: sum(row["test"] == test for row in results) for test in expected_counts}
if counts != expected_counts:
    raise ValueError(f"Số dòng kết quả khác bộ dữ liệu bàn giao: {counts}")

columns = ["test", "scope", "expected", "actual", "status", "bad_grains"]
(sql_dir / "02_test.sql").write_text(full_sql_text, encoding="utf-8")
with (output_dir / "ket_qua_sql.csv").open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(results)

lines = ["Kết quả kiểm thử ETL 4.4 — T02 dùng cent USD", " | ".join(columns)]
lines.extend(" | ".join(str(row.get(c, "")) for c in columns) for row in results)
lines.append(f"Tổng cộng: {len(results)} dòng kết quả")
log = "\n".join(lines) + "\n"
(output_dir / "nhat_ky_kiem_thu.txt").write_text(log, encoding="utf-8")
print(log)


Kết quả kiểm thử ETL 4.4 — T02 dùng cent USD
test | scope | expected | actual | status | bad_grains
T01 | POS toàn bộ | 149116 | 115333 | FAIL | 
T01 | NB04: 41 đồ uống | 115333 | 115333 | PASS | 
T01 | Fact_Inventory_Audit | 1755 | 1755 | PASS | 
T01 | Fact_Waste_Log | 5042 | 5042 | PASS | 
T01 | Bridge_BOM | 178 | 178 | PASS | 
T02 | POS toàn bộ | 69881233 | 53877440 | FAIL | 
T02 | NB04: 41 đồ uống | 53877440 | 53877440 | PASS | 
T03 | POS toàn bộ | 214470 | 176444 | FAIL | 
T03 | NB04: 41 đồ uống | 176444 | 176444 | PASS | 
T04 | Fact_Sales.date_key | 0 | 0 | PASS | 
T04 | Fact_Sales.store_key | 0 | 0 | PASS | 
T04 | Fact_Sales.shift_key | 0 | 0 | PASS | 
T04 | Fact_Sales.menu_item_key | 0 | 0 | PASS | 
T04 | Fact_Inventory_Audit.week_start_date_key | 0 | 0 | PASS | 
T04 | Fact_Inventory_Audit.store_key | 0 | 0 | PASS | 
T04 | Fact_Inventory_Audit.ingredient_key | 0 | 0 | PASS | 
T04 | Fact_Waste_Log.date_key | 0 | 0 | PASS | 
T04 | Fact_Waste_Log.store_key | 0 | 0 | PASS | 
T04 | 

In [ ]:
import zipfile
from google.colab import files

zip_path = Path("/content/Ket_qua_4.4_Colab.zip")
members = ("sql/02_test.sql", "ket_qua_sql.csv", "nhat_ky_kiem_thu.txt")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for member in members:
        archive.write(output_dir / member, member)
print(f"Đã đóng gói: {zip_path.name}")
files.download(str(zip_path))
